# SPECTRA Example 01: Basic Waveform Generation

An introduction to generating and visualizing RF waveforms with SPECTRA.

**Level: Novice**

In this notebook you will learn how to:

- Generate digital modulation waveforms (BPSK, QPSK, 16QAM, FSK)
- Visualize IQ time-domain signals
- Plot constellation diagrams
- View power spectral density (PSD)

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
import spectra as sp
from plot_helpers import plot_constellation, plot_iq_time, plot_psd, savefig

## 1. Generate Your First Waveform

In [ ]:
waveform = sp.QPSK(samples_per_symbol=8, rolloff=0.35)
sample_rate = 1e6  # 1 MHz
iq = waveform.generate(num_symbols=512, sample_rate=sample_rate, seed=42)

print(f"Waveform: {waveform.label}")
print(f"IQ shape: {iq.shape}, dtype: {iq.dtype}")
print(f"Bandwidth: {waveform.bandwidth(sample_rate) / 1e3:.1f} kHz")

## 2. Visualize IQ Time-Domain

In [ ]:
plot_iq_time(iq, title="QPSK \u2014 Time Domain")
savefig("01_qpsk_time.png")

## 3. Constellation Diagram

In [ ]:
plot_constellation(iq, title="QPSK \u2014 Constellation")
savefig("01_qpsk_constellation.png")

## 4. Power Spectral Density

In [ ]:
plot_psd(iq, sample_rate, title="QPSK \u2014 Power Spectral Density")
savefig("01_qpsk_psd.png")

## 5. Compare Multiple Modulation Schemes

In [ ]:
waveforms = [
    sp.BPSK(),
    sp.QPSK(),
    sp.QAM16(),
    sp.QAM64(),
    sp.PSK8(),
    sp.OOK(),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, wf in zip(axes.flat, waveforms):
    iq_i = wf.generate(num_symbols=512, sample_rate=sample_rate, seed=42)
    pts = iq_i[: min(2000, len(iq_i))]
    ax.scatter(pts.real, pts.imag, s=2, alpha=0.4)
    ax.set_title(wf.label)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
fig.suptitle("Constellation Diagrams \u2014 Digital Modulation Comparison", fontsize=14)
fig.tight_layout()
savefig("01_constellation_grid.png")

## 6. PSD Comparison

In [ ]:
waveforms_psd = [
    ("BPSK", sp.BPSK()),
    ("QPSK", sp.QPSK()),
    ("16QAM", sp.QAM16()),
    ("OFDM-64", sp.OFDM()),
    ("FSK", sp.FSK()),
    ("GMSK", sp.GMSK()),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
nfft = 1024
freqs = np.fft.fftshift(np.fft.fftfreq(nfft, d=1.0 / sample_rate))
for ax, (name, wf) in zip(axes.flat, waveforms_psd):
    iq_i = wf.generate(num_symbols=512, sample_rate=sample_rate, seed=42)
    spectrum = np.fft.fftshift(np.fft.fft(iq_i[:nfft], n=nfft))
    psd = 10 * np.log10(np.abs(spectrum) ** 2 + 1e-12)
    ax.plot(freqs / 1e3, psd, linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Freq (kHz)")
    ax.set_ylabel("Power (dB)")
    ax.grid(True, alpha=0.3)
fig.suptitle("Power Spectral Density \u2014 Modulation Comparison", fontsize=14)
fig.tight_layout()
savefig("01_psd_grid.png")

## 7. Analog Modulations

In [ ]:
analog_waveforms = [
    ("AM-DSB", sp.AMDSB()),
    ("AM-SSB (USB)", sp.AMUSB()),
    ("FM", sp.FM()),
    ("Tone", sp.Tone()),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, wf) in zip(axes.flat, analog_waveforms):
    iq_i = wf.generate(num_symbols=1024, sample_rate=sample_rate, seed=42)
    spectrum = np.fft.fftshift(np.fft.fft(iq_i[:nfft], n=nfft))
    psd = 10 * np.log10(np.abs(spectrum) ** 2 + 1e-12)
    ax.plot(freqs / 1e3, psd, linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Freq (kHz)")
    ax.set_ylabel("Power (dB)")
    ax.grid(True, alpha=0.3)
fig.suptitle("Analog Modulation Spectra", fontsize=14)
fig.tight_layout()
savefig("01_analog_psd.png")

plt.close("all")
print("\nDone! Check examples/outputs/ for saved figures.")

## Summary

In this notebook you learned how to:

- Create digital modulation waveforms (BPSK, QPSK, 16QAM, 64QAM, 8PSK, OOK) with a single line of code
- Generate complex IQ samples using the `.generate()` method
- Visualize signals in the time domain, constellation space, and frequency domain
- Compare spectral characteristics of digital and analog modulation schemes

Next: **02 \u2014 Impairments and Channels**